# Evaluating LLM Performance

Christopher La Valle

---

This notebook covers the major approaches to evaluating large language models:

1. **Intrinsic metrics** — perplexity, entropy, token-level accuracy
2. **Text generation quality** — BLEU, ROUGE, METEOR, BERTScore
3. **Task benchmarks** — MMLU, HellaSwag, TruthfulQA
4. **LLM-as-judge** — using a model to score model outputs
5. **Calibration** — does the model know what it doesn't know?
6. **Safety & refusal evaluation**
7. **Latency & throughput profiling**

In [2]:
import math
import re
import time
import json
from collections import Counter
from itertools import islice

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

---
## 1 — Intrinsic Metrics

### 1.1 Perplexity

Perplexity measures how surprised a model is by a held-out text — lower is better.  
It is the exponentiated average negative log-likelihood per token:

$$\text{PPL}(X) = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(x_i \mid x_{<i})\right)$$

A perplexity of $k$ means the model is as confused as if it were choosing uniformly over $k$ tokens at each step.  
State-of-the-art LLMs reach **single-digit perplexity** on standard corpora (WikiText-103, Penn Treebank).

> **Limitation:** perplexity only measures fit to a reference distribution — a model can have low perplexity while still generating factually wrong or harmful text.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


def compute_perplexity(
    text: str, model, tokenizer, stride: int = 512, max_length: int = 1024
) -> float:
    """
    Sliding-window perplexity to handle texts longer than the context window.
    Uses overlapping windows so every token (except the first stride) is
    evaluated with full left context.
    """
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids
    seq_len = input_ids.shape[1]

    nlls, n_tokens = [], 0
    prev_end = 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        target_len = end - prev_end
        chunk = input_ids[:, begin:end]
        target = chunk.clone()
        target[:, :-target_len] = -100  # mask context tokens from loss
        with torch.no_grad():
            loss = model(chunk, labels=target).loss
        nlls.append(loss.item() * target_len)
        n_tokens += target_len
        prev_end = end
        if end == seq_len:
            break

    return math.exp(sum(nlls) / n_tokens)


model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
lm = AutoModelForCausalLM.from_pretrained(model_name)
lm.eval()

texts = {
    "In-domain (news)": (
        "The Federal Reserve raised interest rates by 25 basis points on Wednesday, "
        "citing persistent inflation and a resilient labour market. Officials signalled "
        "that further increases remain possible depending on incoming data."
    ),
    "Out-of-domain (code)": (
        "def quicksort(arr): return arr if len(arr) <= 1 else "
        "quicksort([x for x in arr[1:] if x <= arr[0]]) + [arr[0]] + "
        "quicksort([x for x in arr[1:] if x > arr[0]])"
    ),
    "Random tokens": "xkq zzp mrb vvy wlt oqf jjn",
}

ppls = {}
for label, text in texts.items():
    ppl = compute_perplexity(text, lm, tokenizer)
    ppls[label] = ppl
    print(f"{label:30s} PPL = {ppl:.1f}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


In-domain (news)               PPL = 36.4
Out-of-domain (code)           PPL = 7.6
Random tokens                  PPL = 222.1


In [ ]:
fig = go.Figure(
    go.Bar(
        x=list(ppls.keys()),
        y=list(ppls.values()),
        marker_color=px.colors.qualitative.Plotly[:3],
        text=[f"{v:.1f}" for v in ppls.values()],
        textposition="outside",
    )
)
fig.update_layout(
    title="GPT-2 Perplexity Across Text Types",
    yaxis_title="Perplexity (lower = better)",
    template="plotly_dark",
    height=380,
)
fig.show()

### 1.2 Bits-Per-Character / Bits-Per-Token

$$\text{BPC} = \frac{\log_2 \text{PPL}}{\text{avg chars per token}}$$

BPC normalises for tokenizer differences — useful when comparing models with different vocabularies.

In [ ]:
def bits_per_char(ppl: float, text: str, tokenizer) -> float:
    n_tokens = len(tokenizer.encode(text))
    n_chars = len(text)
    avg_chars_per_token = n_chars / n_tokens
    return math.log2(ppl) / avg_chars_per_token


for label, text in texts.items():
    bpc = bits_per_char(ppls[label], text, tokenizer)
    print(f"{label:30s} BPC = {bpc:.3f}")

---
## 2 — Text Generation Quality

### 2.1 BLEU (Bilingual Evaluation Understudy)

Designed for machine translation (Papineni et al., 2002). Measures n-gram **precision** between a hypothesis and one or more reference texts, with a brevity penalty for short outputs:

$$\text{BLEU} = BP \cdot \exp\!\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

where $p_n$ is the clipped n-gram precision and $BP = \min(1, e^{1-r/c})$.

> **Limitations:** ignores recall, synonyms, and paraphrasing. Weak correlation with human judgement for open-ended generation.

In [ ]:
def ngrams(tokens: list[str], n: int) -> Counter:
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def bleu_score(hypothesis: str, references: list[str], max_n: int = 4) -> dict:
    hyp_tokens = hypothesis.lower().split()
    ref_tokens_list = [r.lower().split() for r in references]

    # Brevity penalty
    c = len(hyp_tokens)
    r = min((len(rt) for rt in ref_tokens_list), key=lambda l: abs(l - c))
    bp = 1.0 if c >= r else math.exp(1 - r / c)

    precisions = []
    for n in range(1, max_n + 1):
        hyp_ng = ngrams(hyp_tokens, n)
        clipped = Counter()
        for ng, cnt in hyp_ng.items():
            max_ref = max(ngrams(rt, n).get(ng, 0) for rt in ref_tokens_list)
            clipped[ng] = min(cnt, max_ref)
        denom = max(1, sum(hyp_ng.values()))
        precisions.append(sum(clipped.values()) / denom)

    log_avg = sum(math.log(p) if p > 0 else -1e9 for p in precisions) / max_n
    bleu = bp * math.exp(log_avg)
    return {"bleu": bleu, "bp": bp, "precisions": precisions}


reference = "The cat sat on the mat near the window"
hypotheses = [
    ("Perfect match", "The cat sat on the mat near the window"),
    ("Near match", "The cat sat on a mat near the window"),
    ("Partial overlap", "A cat rested on the mat"),
    ("Off topic", "Dogs love to run in the park all day"),
]

results = []
for label, hyp in hypotheses:
    r = bleu_score(hyp, [reference])
    results.append(
        {
            "label": label,
            "bleu": r["bleu"],
            **{f"p{i+1}": r["precisions"][i] for i in range(4)},
        }
    )
    print(
        f"{label:20s}  BLEU={r['bleu']:.3f}  "
        f"p1={r['precisions'][0]:.2f}  p2={r['precisions'][1]:.2f}  "
        f"p3={r['precisions'][2]:.2f}  p4={r['precisions'][3]:.2f}"
    )

In [ ]:
labels = [r["label"] for r in results]
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["BLEU Score", "N-gram Precision Breakdown"]
)

fig.add_trace(
    go.Bar(
        x=labels,
        y=[r["bleu"] for r in results],
        marker_color=px.colors.qualitative.Plotly[:4],
        showlegend=False,
    ),
    row=1,
    col=1,
)

for i, n in enumerate(["p1", "p2", "p3", "p4"], 1):
    fig.add_trace(
        go.Bar(name=f"{i}-gram", x=labels, y=[r[n] for r in results]), row=1, col=2
    )

fig.update_layout(
    barmode="group", template="plotly_dark", height=400, title="BLEU Score Analysis"
)
fig.show()

### 2.2 ROUGE (Recall-Oriented Understudy for Gisting Evaluation)

Designed for summarisation (Lin, 2004). Unlike BLEU, ROUGE emphasises **recall**.

| Variant | What it measures |
|---|---|
| **ROUGE-N** | n-gram recall between hypothesis and reference |
| **ROUGE-L** | Longest Common Subsequence (LCS) F1 |
| **ROUGE-W** | Weighted LCS (rewards consecutive matches) |
| **ROUGE-S** | Skip-bigram co-occurrence |

In [ ]:
def rouge_n(hypothesis: str, reference: str, n: int = 2) -> dict:
    hyp = hypothesis.lower().split()
    ref = reference.lower().split()
    hyp_ng = ngrams(hyp, n)
    ref_ng = ngrams(ref, n)
    overlap = sum(min(hyp_ng[g], ref_ng[g]) for g in hyp_ng)
    recall = overlap / max(1, sum(ref_ng.values()))
    precision = overlap / max(1, sum(hyp_ng.values()))
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


def lcs_length(a: list, b: list) -> int:
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = (
                dp[i - 1][j - 1] + 1
                if a[i - 1] == b[j - 1]
                else max(dp[i - 1][j], dp[i][j - 1])
            )
    return dp[m][n]


def rouge_l(hypothesis: str, reference: str) -> dict:
    hyp = hypothesis.lower().split()
    ref = reference.lower().split()
    lcs = lcs_length(hyp, ref)
    recall = lcs / max(1, len(ref))
    precision = lcs / max(1, len(hyp))
    f1 = 2 * precision * recall / max(1e-9, precision + recall)
    return {"precision": precision, "recall": recall, "f1": f1}


print(f"{'Label':20s}  {'ROUGE-1 F1':>10}  {'ROUGE-2 F1':>10}  {'ROUGE-L F1':>10}")
print("-" * 60)
for label, hyp in hypotheses:
    r1 = rouge_n(hyp, reference, 1)["f1"]
    r2 = rouge_n(hyp, reference, 2)["f1"]
    rl = rouge_l(hyp, reference)["f1"]
    print(f"{label:20s}  {r1:10.3f}  {r2:10.3f}  {rl:10.3f}")

### 2.3 METEOR & BERTScore

**METEOR** (Banerjee & Lavie, 2005) improves on BLEU by incorporating:
- Stemming (e.g. `run` ↔ `running`)
- Synonym matching via WordNet
- Chunk-based fragmentation penalty (rewards word order)

**BERTScore** (Zhang et al., 2020) replaces n-gram overlap with contextual embeddings:

$$P_{\text{BERT}} = \frac{1}{|\hat{y}|}\sum_{\hat{y}_j \in \hat{y}} \max_{y_i \in y} \cos(\mathbf{e}_{\hat{y}_j}, \mathbf{e}_{y_i})$$

BERTScore captures semantic similarity that surface-form metrics miss (synonyms, paraphrases).

In [ ]:
from transformers import AutoTokenizer as AT, AutoModel
import torch.nn.functional as F


def bert_score(
    hypothesis: str, reference: str, model_name: str = "bert-base-uncased"
) -> dict:
    tok = AT.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name)
    mdl.eval()

    def embed(text):
        enc = tok(text, return_tensors="pt", truncation=True, max_length=128)
        with torch.no_grad():
            out = mdl(**enc)
        # token embeddings (exclude [CLS] and [SEP])
        return out.last_hidden_state[0, 1:-1]

    h_emb = F.normalize(embed(hypothesis), dim=-1)  # (H, d)
    r_emb = F.normalize(embed(reference), dim=-1)  # (R, d)

    sim = h_emb @ r_emb.T  # (H, R)
    precision = sim.max(dim=1).values.mean().item()
    recall = sim.max(dim=0).values.mean().item()
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    return {"precision": precision, "recall": recall, "f1": f1}


print(f"{'Label':20s}  {'BERTScore P':>11}  {'BERTScore R':>11}  {'BERTScore F1':>12}")
print("-" * 65)
for label, hyp in hypotheses:
    s = bert_score(hyp, reference)
    print(f"{label:20s}  {s['precision']:11.3f}  {s['recall']:11.3f}  {s['f1']:12.3f}")

### 2.4 Metric Comparison

Different metrics tell different stories. Visualising them together reveals their trade-offs.

In [ ]:
metric_data = {}
for label, hyp in hypotheses:
    bleu = bleu_score(hyp, [reference])["bleu"]
    r1 = rouge_n(hyp, reference, 1)["f1"]
    r2 = rouge_n(hyp, reference, 2)["f1"]
    rl = rouge_l(hyp, reference)["f1"]
    bs = bert_score(hyp, reference)["f1"]
    metric_data[label] = {
        "BLEU": bleu,
        "ROUGE-1": r1,
        "ROUGE-2": r2,
        "ROUGE-L": rl,
        "BERTScore": bs,
    }

metrics = ["BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore"]
fig = go.Figure()
for label, scores in metric_data.items():
    fig.add_trace(
        go.Scatterpolar(
            r=[scores[m] for m in metrics] + [scores[metrics[0]]],
            theta=metrics + [metrics[0]],
            fill="toself",
            name=label,
        )
    )
fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1])),
    title="Metric Comparison — Radar Chart",
    template="plotly_dark",
    height=480,
)
fig.show()

---
## 3 — Task Benchmarks

Academic benchmarks measure **capability** on held-out tasks rather than generation quality.

| Benchmark | Task type | # questions | What it tests |
|---|---|---|---|
| **MMLU** | Multiple choice | 14,042 | World knowledge across 57 subjects |
| **HellaSwag** | Sentence completion | 10,042 | Common-sense reasoning |
| **TruthfulQA** | Open generation | 817 | Factual accuracy, avoidance of falsehoods |
| **GSM8K** | Math word problems | 1,319 | Multi-step arithmetic reasoning |
| **HumanEval** | Code completion | 164 | Functional correctness of generated code |
| **BIG-Bench Hard** | Mixed | 6,511 | Tasks that challenge large models |

### 3.1 Evaluating Multiple-Choice Benchmarks (MMLU-style)

The standard evaluation uses **log-likelihood scoring**: compute the LM log-probability of each answer continuation and pick the argmax — no greedy decoding needed.

In [ ]:
def score_choice(prompt: str, choice: str, model, tokenizer) -> float:
    """Log-likelihood of `choice` given `prompt`."""
    full = prompt + choice
    enc_full = tokenizer(full, return_tensors="pt")
    enc_prompt = tokenizer(prompt, return_tensors="pt")
    n_prompt = enc_prompt.input_ids.shape[1]

    with torch.no_grad():
        logits = model(**enc_full).logits[0]  # (seq, vocab)

    log_probs = torch.log_softmax(logits, dim=-1)
    ids = enc_full.input_ids[0]
    # Sum log-probs over the choice tokens only
    choice_log_prob = sum(
        log_probs[i - 1, ids[i]].item() for i in range(n_prompt, len(ids))
    )
    return choice_log_prob


def evaluate_mcq(
    question: str, choices: list[str], answer_idx: int, model, tokenizer
) -> dict:
    scores = [score_choice(question + "\n", c, model, tokenizer) for c in choices]
    pred = int(np.argmax(scores))
    return {
        "scores": scores,
        "pred": pred,
        "correct": pred == answer_idx,
        "probs": torch.softmax(torch.tensor(scores), dim=0).tolist(),
    }


# Sample MMLU-style questions
mmlu_sample = [
    {
        "question": "Which of the following best describes the function of mitochondria?",
        "choices": [
            "A) Protein synthesis",
            "B) ATP production",
            "C) DNA replication",
            "D) Lipid storage",
        ],
        "answer": 1,
    },
    {
        "question": "What is the capital of France?",
        "choices": ["A) Berlin", "B) Madrid", "C) Paris", "D) Rome"],
        "answer": 2,
    },
    {
        "question": "Which sorting algorithm has worst-case O(n log n) time complexity?",
        "choices": [
            "A) Bubble sort",
            "B) Insertion sort",
            "C) Quick sort",
            "D) Merge sort",
        ],
        "answer": 3,
    },
]

correct = 0
for item in mmlu_sample:
    result = evaluate_mcq(
        item["question"], item["choices"], item["answer"], lm, tokenizer
    )
    mark = "✓" if result["correct"] else "✗"
    correct += result["correct"]
    print(f"{mark} Q: {item['question'][:55]}...")
    print(
        f"    Pred={item['choices'][result['pred']]}  "
        f"Gold={item['choices'][item['answer']]}"
    )
    print(f"    Probs: {[f'{p:.2f}' for p in result['probs']]}")

print(f"\nAccuracy: {correct}/{len(mmlu_sample)} = {correct/len(mmlu_sample):.0%}")

### 3.2 Few-Shot vs Zero-Shot Evaluation

GPT-3 popularised **in-context learning**: prepending $k$ labeled examples to the prompt without weight updates.

| Setting | Examples in prompt | Notes |
|---|---|---|
| Zero-shot | 0 | Task described in natural language only |
| One-shot | 1 | Single demonstration |
| Few-shot | 2–32 | Balance capability vs context length |
| Chain-of-thought | ≥1 with reasoning steps | Dramatically improves multi-step tasks |

In [ ]:
def build_few_shot_prompt(
    question: str, choices: list[str], examples: list[dict], k: int = 2
) -> str:
    prompt = ""
    for ex in examples[:k]:
        letters = "ABCD"
        prompt += f"Q: {ex['question']}\n"
        for i, c in enumerate(ex["choices"]):
            prompt += f"  {letters[i]}) {c}\n"
        prompt += f"A: {letters[ex['answer']]}\n\n"
    prompt += f"Q: {question}\n"
    for c in choices:
        prompt += f"  {c}\n"
    prompt += "A:"
    return prompt


test_q = mmlu_sample[2]
examples = mmlu_sample[:2]
prompt_0 = f"Q: {test_q['question']}\nA:"
prompt_2 = build_few_shot_prompt(test_q["question"], test_q["choices"], examples, k=2)

print("=== Zero-shot prompt ===")
print(prompt_0)
print("\n=== 2-shot prompt ===")
print(prompt_2)

---
## 4 — LLM-as-Judge

For open-ended tasks (creative writing, instruction following, summarisation) automatic metrics correlate poorly with human preferences.  
**LLM-as-judge** uses a capable model to score outputs — the approach behind **MT-Bench**, **Alpaca Eval**, and **Arena-Hard**.

**Common protocols:**
- **Pointwise scoring** — judge scores each response on a scale (1–10)
- **Pairwise comparison** — judge picks winner between two responses (A/B)
- **Reference-guided** — judge compares against a gold response

**Known failure modes:**
- Position bias (prefers response A over B regardless of quality)
- Verbosity bias (prefers longer responses)
- Self-enhancement bias (model rates its own outputs higher)

In [ ]:
POINTWISE_TEMPLATE = """\
You are an impartial evaluator. Rate the following response to the given question \
on a scale from 1 (poor) to 10 (excellent).

Criteria: {criteria}

Question: {question}

Response: {response}

Provide your rating as a single integer between 1 and 10, followed by a one-sentence justification.
Rating:"""

PAIRWISE_TEMPLATE = """\
You are an impartial evaluator. Compare the two responses below and decide which is better.

Criteria: {criteria}

Question: {question}

Response A: {response_a}

Response B: {response_b}

Which response is better? Answer with exactly one of: A, B, or Tie. Then give a one-sentence reason.
Answer:"""


def parse_pointwise(output: str) -> int | None:
    match = re.search(r"\b([1-9]|10)\b", output)
    return int(match.group(1)) if match else None


def parse_pairwise(output: str) -> str | None:
    match = re.search(r"\b(A|B|Tie)\b", output)
    return match.group(1) if match else None


# Demonstrate template construction (real usage requires an API call or local model)
sample_question = "Explain the difference between precision and recall."
sample_response_a = (
    "Precision is the fraction of retrieved items that are relevant. "
    "Recall is the fraction of relevant items that are retrieved."
)
sample_response_b = "Precision means how accurate your results are."

print(
    POINTWISE_TEMPLATE.format(
        criteria="Correctness, clarity, and completeness",
        question=sample_question,
        response=sample_response_a,
    )
)
print("\n" + "=" * 60)
print(
    PAIRWISE_TEMPLATE.format(
        criteria="Correctness, clarity, and completeness",
        question=sample_question,
        response_a=sample_response_a,
        response_b=sample_response_b,
    )
)

In [ ]:
# Simulate position bias analysis across N pairwise evaluations
np.random.seed(0)
n_evals = 100

# Suppose true winner is always A, but judge sometimes flips due to position bias
bias_levels = {"No bias": 0.0, "Mild bias": 0.15, "Strong bias": 0.35}
fig = go.Figure()

for label, bias in bias_levels.items():
    # judge votes A correctly with prob (1-bias), flips to B with prob bias
    votes = np.random.choice(["A", "B"], size=n_evals, p=[1 - bias, bias])
    a_wins = (votes == "A").cumsum() / (np.arange(n_evals) + 1)
    fig.add_trace(
        go.Scatter(x=np.arange(1, n_evals + 1), y=a_wins, mode="lines", name=label)
    )

fig.add_hline(
    y=0.5, line_dash="dash", line_color="red", annotation_text="Random chance"
)
fig.update_layout(
    title="LLM Judge Position Bias Simulation",
    xaxis_title="Number of evaluations",
    yaxis_title="Win rate for true winner (A)",
    template="plotly_dark",
    height=400,
)
fig.show()

---
## 5 — Calibration

A well-calibrated model's **confidence** (probability) should match its **accuracy**.  
If a model says "I'm 90% confident" on 100 questions, it should get ~90 right.

**Expected Calibration Error (ECE):**
$$\text{ECE} = \sum_{b=1}^{B} \frac{|\mathcal{B}_b|}{N} \left| \text{acc}(\mathcal{B}_b) - \text{conf}(\mathcal{B}_b) \right|$$

Buckets $\mathcal{B}_b$ partition predictions by confidence interval. Perfect calibration → ECE = 0.

LLMs tend to be **overconfident** — their token probabilities often don't reflect true uncertainty.

In [ ]:
def expected_calibration_error(
    confidences: np.ndarray, correct: np.ndarray, n_bins: int = 10
) -> float:
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(confidences)
    for lo, hi in zip(bins, bins[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        acc = correct[mask].mean()
        conf = confidences[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return ece


def reliability_diagram(
    confidences: np.ndarray, correct: np.ndarray, n_bins: int = 10, title: str = ""
) -> go.Figure:
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers, bin_accs, bin_confs, bin_sizes = [], [], [], []
    for lo, hi in zip(bins, bins[1:]):
        mask = (confidences >= lo) & (confidences < hi)
        if mask.sum() == 0:
            continue
        bin_centers.append((lo + hi) / 2)
        bin_accs.append(correct[mask].mean())
        bin_confs.append(confidences[mask].mean())
        bin_sizes.append(mask.sum())

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=[0, 1],
            y=[0, 1],
            mode="lines",
            line=dict(dash="dash", color="gray"),
            name="Perfect",
        )
    )
    fig.add_trace(
        go.Bar(
            x=bin_centers,
            y=bin_accs,
            width=0.08,
            name="Accuracy",
            marker_color="steelblue",
            opacity=0.8,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=bin_confs,
            y=bin_accs,
            mode="markers+lines",
            marker=dict(size=10, color="orange"),
            name="Calibration curve",
        )
    )
    ece = expected_calibration_error(confidences, correct, n_bins)
    fig.update_layout(
        title=f"{title}  (ECE = {ece:.3f})",
        xaxis_title="Confidence",
        yaxis_title="Accuracy",
        xaxis=dict(range=[0, 1]),
        yaxis=dict(range=[0, 1]),
        template="plotly_dark",
        height=400,
    )
    return fig


rng = np.random.default_rng(7)
n = 500

# Overconfident model
true_conf = rng.uniform(0.4, 1.0, n)
correct = rng.binomial(1, true_conf)  # actual correctness
over_conf = np.clip(true_conf + rng.normal(0.15, 0.05, n), 0.01, 0.99)

fig = reliability_diagram(over_conf, correct.astype(float), title="Overconfident Model")
fig.show()

# Well-calibrated model (confidence ≈ accuracy)
good_conf = true_conf + rng.normal(0.0, 0.02, n)
good_conf = np.clip(good_conf, 0.01, 0.99)
reliability_diagram(
    good_conf, correct.astype(float), title="Well-Calibrated Model"
).show()

### 5.1 Temperature Scaling

**Temperature scaling** is the simplest post-hoc calibration method:  
divide logits by a learned scalar $T$ before softmax.

- $T > 1$ → softer distribution (less confident)
- $T < 1$ → sharper distribution (more confident)
- $T = 1$ → no change

Find optimal $T$ by minimising NLL on a calibration set (held-out from training).

In [ ]:
def apply_temperature(logits: np.ndarray, T: float) -> np.ndarray:
    e = np.exp((logits - logits.max()) / T)
    return e / e.sum()


temperatures = [0.5, 1.0, 1.5, 2.0, 3.0]
logits = np.array([3.0, 1.5, 0.5, -0.5, -1.5])  # example logit vector

fig = go.Figure()
for T in temperatures:
    probs = apply_temperature(logits, T)
    fig.add_trace(go.Bar(name=f"T={T}", x=[f"token {i}" for i in range(5)], y=probs))

fig.update_layout(
    barmode="group",
    title="Effect of Temperature on Output Distribution",
    yaxis_title="Probability",
    template="plotly_dark",
    height=400,
)
fig.show()

---
## 6 — Safety & Refusal Evaluation

Safety evaluations test whether a model:
1. **Refuses** clearly harmful requests (attack success rate ↓)
2. **Does not refuse** benign requests (false refusal rate ↓)
3. **Resists jailbreaks** — adversarial prompts designed to bypass safety training

Key metrics:

| Metric | Formula | Goal |
|---|---|---|
| Attack Success Rate (ASR) | harmful completions / harmful requests | ↓ |
| False Refusal Rate (FRR) | refused benign / total benign | ↓ |
| Over-refusal score | 1 − FRR | ↑ |
| Safety F1 | harmonic mean of (1−ASR) and (1−FRR) | ↑ |

In [ ]:
def safety_metrics(tp: int, fp: int, tn: int, fn: int) -> dict:
    """
    tp = correctly refused harmful
    fp = incorrectly refused benign (false refusal)
    tn = correctly answered benign
    fn = harmful request not refused (attack succeeded)
    """
    n_harmful = tp + fn
    n_benign = tn + fp
    asr = fn / max(1, n_harmful)
    frr = fp / max(1, n_benign)
    safety_score = 1 - asr
    helpfulness = 1 - frr
    f1 = 2 * safety_score * helpfulness / max(1e-9, safety_score + helpfulness)
    return {
        "ASR": asr,
        "FRR": frr,
        "Safety": safety_score,
        "Helpfulness": helpfulness,
        "Safety-F1": f1,
    }


models_safety = {
    "Overly cautious": safety_metrics(tp=98, fp=40, tn=60, fn=2),
    "Balanced": safety_metrics(tp=92, fp=8, tn=92, fn=8),
    "Under-restricted": safety_metrics(tp=60, fp=2, tn=98, fn=40),
}

metrics_list = ["ASR", "FRR", "Safety", "Helpfulness", "Safety-F1"]
fig = go.Figure()
for model_label, m in models_safety.items():
    fig.add_trace(
        go.Bar(name=model_label, x=metrics_list, y=[m[k] for k in metrics_list])
    )
fig.add_hline(y=0.5, line_dash="dot", line_color="gray")
fig.update_layout(
    barmode="group",
    title="Safety vs Helpfulness Trade-off",
    yaxis_title="Score (ASR/FRR lower is better, rest higher is better)",
    template="plotly_dark",
    height=420,
)
fig.show()

---
## 7 — Latency & Throughput Profiling

Practical deployment requires understanding performance beyond accuracy:

| Metric | Definition |
|---|---|
| **Time to First Token (TTFT)** | Latency until the first output token (prefill phase) |
| **Time per Output Token (TPOT)** | Decode phase throughput |
| **Tokens per second (TPS)** | Total output tokens / total time |
| **Requests per second (RPS)** | Batch efficiency |

Memory usage scales as: $\text{KV cache} \approx 2 \times n_{\text{layers}} \times n_{\text{heads}} \times d_{\text{head}} \times \text{seq\_len} \times \text{batch} \times \text{dtype\_bytes}$

In [ ]:
def profile_generation(prompt: str, model, tokenizer, max_new_tokens: int = 50) -> dict:
    inputs = tokenizer(prompt, return_tensors="pt")
    n_input = inputs.input_ids.shape[1]

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    t1 = time.perf_counter()

    n_output = output_ids.shape[1] - n_input
    total_time = t1 - t0
    return {
        "prompt_tokens": n_input,
        "output_tokens": n_output,
        "total_time_s": total_time,
        "tokens_per_sec": n_output / total_time,
        "ms_per_token": total_time / n_output * 1000,
    }


prompts = [
    ("Short", "The capital of France is"),
    (
        "Medium",
        "Explain the difference between supervised and unsupervised learning in machine learning.",
    ),
    (
        "Long",
        " ".join(["The history of artificial intelligence spans many decades."] * 8),
    ),
]

profiles = []
for label, prompt in prompts:
    p = profile_generation(prompt, lm, tokenizer, max_new_tokens=40)
    p["label"] = label
    profiles.append(p)
    print(
        f"{label:8s} | input={p['prompt_tokens']:4d} tokens | "
        f"output={p['output_tokens']:3d} | "
        f"{p['tokens_per_sec']:5.1f} tok/s | "
        f"{p['ms_per_token']:5.1f} ms/tok"
    )

In [ ]:
fig = make_subplots(
    rows=1, cols=2, subplot_titles=["Tokens per Second", "ms per Token"]
)
labels = [p["label"] for p in profiles]
colors = px.colors.qualitative.Plotly

fig.add_trace(
    go.Bar(
        x=labels,
        y=[p["tokens_per_sec"] for p in profiles],
        marker_color=colors[:3],
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=labels,
        y=[p["ms_per_token"] for p in profiles],
        marker_color=colors[:3],
        showlegend=False,
    ),
    row=1,
    col=2,
)

fig.update_layout(
    title="Latency Profiling — GPT-2 (CPU)", template="plotly_dark", height=380
)
fig.show()

In [ ]:
# KV cache memory estimation
def kv_cache_gb(
    n_layers: int,
    n_heads: int,
    d_head: int,
    seq_len: int,
    batch_size: int,
    dtype_bytes: int = 2,
) -> float:
    """Estimate KV cache size in GB (2 for K and V)."""
    return (2 * n_layers * n_heads * d_head * seq_len * batch_size * dtype_bytes) / 1e9


# Compare across model sizes at seq_len=4096, batch=1
configs = {
    "GPT-2 (117M)": dict(n_layers=12, n_heads=12, d_head=64),
    "GPT-2 XL (1.5B)": dict(n_layers=48, n_heads=25, d_head=64),
    "LLaMA-7B": dict(n_layers=32, n_heads=32, d_head=128),
    "LLaMA-70B": dict(n_layers=80, n_heads=64, d_head=128),
}
seq_lengths = [512, 1024, 2048, 4096, 8192]

fig = go.Figure()
for label, cfg in configs.items():
    mem = [kv_cache_gb(**cfg, seq_len=s, batch_size=1) for s in seq_lengths]
    fig.add_trace(go.Scatter(x=seq_lengths, y=mem, mode="lines+markers", name=label))

fig.update_layout(
    title="KV Cache Memory vs Sequence Length (batch=1, float16)",
    xaxis_title="Sequence length",
    yaxis_title="Memory (GB)",
    template="plotly_dark",
    height=420,
)
fig.show()

---
## Summary

| Evaluation axis | Key methods | Tools |
|---|---|---|
| **Language modelling** | Perplexity, BPC | `transformers`, custom |
| **Generation quality** | BLEU, ROUGE, METEOR, BERTScore | `evaluate`, `bert-score` |
| **Capabilities** | MMLU, HellaSwag, GSM8K, HumanEval | `lm-evaluation-harness` |
| **Alignment** | LLM-as-judge, MT-Bench, Alpaca Eval | `fastchat`, `alpaca_eval` |
| **Calibration** | ECE, reliability diagram, temperature scaling | custom |
| **Safety** | ASR, FRR, Safety-F1 | `promptbench`, red-teaming |
| **Efficiency** | TTFT, TPS, KV cache memory | profiler, `torch.profiler` |

**Design principles:**
- No single metric captures all dimensions — always use a **suite**.
- Automatic metrics are cheap but imperfect; human evaluation remains the gold standard.
- Benchmark contamination is a real risk — prefer held-out or dynamically generated test sets.
- Calibration matters as much as accuracy for high-stakes applications.
- Safety and helpfulness are complementary, not opposing — measure both.